# Data to Metadata to Dummy Data

In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv(
    "penguin_plus.csv", 
    parse_dates=["timestamp", "timestamp_with_time"]
)
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,39.1,18.7,NaN,3750.0,True,73,2025-04-13,2025-04-13 15:04:00,1
1,Adelie,Torgersen,39.5,17.4,NaN,3800.0,False,1,2025-12-15,2025-12-15 18:15:26,2


In [3]:
import sys
sys.path.append('../')

In [4]:
from csvw_safe.make_metadata_from_data import make_metadata_from_data
from csvw_safe.validate_metadata import validate_metadata
from csvw_safe.validate_metadata_shacl import validate_metadata_shacl
from csvw_safe.make_dummy_from_metadata import make_dummy_from_metadata
from csvw_safe.assert_same_structure import assert_same_structure

### Generate metadata

In [5]:
metadata_path_1 = 'metadata/penguin_metadata_basic.json-ld'
metadata_path_2 = 'metadata/penguin_metadata_continuous_partitions.json-ld'
metadata_path_3 = 'metadata/penguin_metadata_column_groups.json-ld'
metadata_path_4 = 'metadata/penguin_metadata_continuous_partitions_in_column_groups.json-ld'

#### Basic

In [6]:
metadata_1 = make_metadata_from_data(
    df, privacy_unit = "penguin_id"
)
with open(metadata_path_1, 'w', encoding='utf-8') as f:
    json.dump(metadata_1, f)

In [7]:
#metadata_1['csvw:tableSchema']['columns']

#### Known continuous partitions

In [8]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ['2025-01-02 00:00:00', '2025-06-02 00:00:00', '2025-12-30 00:00:00'],
}

In [9]:
metadata_2 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions
)
with open(metadata_path_2, 'w', encoding='utf-8') as f:
    json.dump(metadata_2, f)

#### Known column groups

In [10]:
column_groups = [
    ["species", "island"],
    ["species", "island", "favourite_number"]
]

In [11]:
metadata_3 = make_metadata_from_data(
    df,
    privacy_unit = "penguin_id", 
    column_groups = column_groups
)
with open(metadata_path_3, 'w', encoding='utf-8') as f:
    json.dump(metadata_3, f)

In [12]:
#metadata_3["csvw-safe:additionalInformation"]

#### Known column groups containing public continuous partitions

In [13]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ['2025-01-02 00:00:00', '2025-06-02 00:00:00', '2025-12-30 00:00:00'],
}
column_groups = [
    ["species", "island"],
    ["species", "island", "bill_length_mm"],
]

In [14]:
metadata_4 = make_metadata_from_data(
    df, 
    privacy_unit = "penguin_id", 
    continuous_partitions = continuous_partitions,
    column_groups = column_groups
)
with open(metadata_path_4, 'w', encoding='utf-8') as f:
    json.dump(metadata_4, f)

/home/onyxia/work/csvw-dp/csvw-safe-library/examples/../csvw_safe/make_metadata_from_data.py:209: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df_copy.groupby(group_by_cols)


### Validate metadata

In [15]:
errors = validate_metadata(metadata_1)
errors

[]

In [16]:
errors = validate_metadata(metadata_2)
errors

[]

In [17]:
errors = validate_metadata(metadata_3)
errors

[]

In [18]:
errors = validate_metadata(metadata_4)
errors

[]

### Validate metadata SHACL

In [19]:
shacl_path = '../../csvw-safe-constraints.ttl'
metadata_path_1

'metadata/penguin_metadata_basic.json-ld'

In [20]:
validate_metadata_shacl(metadata_path_1, shacl_path)

(True, 'Validation Report\nConforms: True\n')

In [21]:
validate_metadata_shacl(metadata_path_2, shacl_path)

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-package

(True, 'Validation Report\nConforms: True\n')

In [22]:
validate_metadata_shacl(metadata_path_3, shacl_path)

(True, 'Validation Report\nConforms: True\n')

In [23]:
validate_metadata_shacl(metadata_path_4, shacl_path)

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-package

(True, 'Validation Report\nConforms: True\n')

### Generate Dummy

In [19]:
dummy_df_1 = make_dummy_from_metadata(metadata_1, nb_rows = 100, seed = 0)
dummy_df_1.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,45.299668,15.771346,194.144271,2970.477702,False,147.415406,NaT,NaT,5
1,Chinstrap,Torgersen,38.490255,19.071391,227.902277,4683.761985,False,47.637644,NaT,NaT,3
2,Chinstrap,Biscoe,54.151716,16.966985,NaN,3390.541733,False,227.316155,NaT,NaT,5
3,Adelie,Biscoe,57.497079,17.362747,204.169551,2942.725394,False,256.734911,NaT,NaT,1
4,Adelie,Torgersen,39.418582,19.733192,177.320726,5483.752553,False,56.904445,NaT,NaT,3


In [20]:
dummy_df_2 = make_dummy_from_metadata(metadata_2, nb_rows = 100, seed = 0)
dummy_df_2.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Gentoo,Dream,NaN,21.281469,199.469663,4767.878768,True,157.471454,NaT,NaT,3
1,Chinstrap,Torgersen,NaN,17.330887,178.233966,3214.807503,False,324.996039,NaT,NaT,3
2,Chinstrap,Biscoe,32.661303,19.428951,186.671510,2749.456291,False,11.411436,NaT,NaT,1
3,Adelie,Biscoe,55.389344,20.776236,227.976356,4262.008408,False,23.543598,NaT,NaT,2
4,Adelie,Torgersen,49.310173,17.099635,NaN,5443.909819,False,10.485224,NaT,NaT,4


In [21]:
dummy_df_3 = make_dummy_from_metadata(metadata_3, nb_rows = 100, seed = 0)
dummy_df_3.head()

,species,island,favourite_number,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time
0,Adelie,Torgersen,0,45.299668,15.771346,194.144271,2970.477702,False,147.415406,NaT,NaT
1,Chinstrap,Dream,5,38.490255,19.071391,227.902277,4683.761985,False,47.637644,NaT,NaT
2,Adelie,Dream,3,54.151716,16.966985,NaN,3390.541733,False,227.316155,NaT,NaT
3,Adelie,Dream,1,57.497079,17.362747,204.169551,2942.725394,False,256.734911,NaT,NaT
4,Chinstrap,Dream,5,39.418582,19.733192,177.320726,5483.752553,False,56.904445,NaT,NaT


In [22]:
dummy_df_3 = make_dummy_from_metadata(metadata_4, nb_rows = 100, seed = 0)
dummy_df_3.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,34.799879,15.785326,NaN,5116.142809,True,157.749159,NaT,NaT,5
1,Chinstrap,Dream,52.323729,14.675065,208.082603,4612.378044,False,258.495274,NaT,NaT,2
2,Adelie,Dream,48.018806,18.749224,178.212002,5728.231266,True,166.428367,NaT,NaT,5
3,Adelie,Dream,39.235302,14.738902,215.131775,4451.473560,False,242.667471,NaT,NaT,5
4,Chinstrap,Dream,52.661303,17.952578,207.296422,4413.401532,True,109.158133,NaT,NaT,3
